# SpamBase Hyperparameter Sweep for Alpha=2.0 Targeted Search


## Why the repair changes the search

The original alpha-trend notebook had useful structure, but model selection remained too accuracy-anchored. This repair adds explicit two-track objectives: (A) best validation accuracy and (B) best validation accuracy near alpha=2.0.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

import xgboost as xgb
import weightwatcher as ww
from xgboost2ww import convert


In [ ]:
RNG = 123
np.random.seed(RNG)

TEST_SIZE = 0.20
VALID_SIZE_FROM_TRAIN = 0.20
BASELINE_SEEDS = [11, 29, 47]

TARGET_ALPHA = 2.0
HARD_ALPHA_BAND = (1.85, 2.15)
SOFT_ALPHA_BAND = (1.70, 2.30)
VAL_ACC_TOL = 0.002
N_COARSE_SEEDS = 3
N_REFINE_SEEDS = 5

W_MATRIX_PRIMARY = 'W7'
W_MATRIX_SECONDARY = 'W8'
RUN_SECONDARY_W8 = False

WW_ANALYZE_OPTIONS = dict(randomize=True, detX=True)
WW_MIN_TAIL = 10

BASE_OBJECTIVE_PARAMS = {'objective': 'binary:logistic', 'eval_metric': ['logloss', 'auc'], 'tree_method': 'hist'}

RESULTS_DIR = Path('./results/spambase_alpha2_targeted')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_spambase_data():
    try:
        ds = fetch_openml(name='spambase', version=1, as_frame=False, parser='auto')
        X = ds.data.astype(np.float32)
        y = np.asarray(ds.target)
        if y.dtype.kind in 'OUS':
            y = np.where(np.isin(y.astype(str), ['1', 'spam', 'yes', 'True']), 1, 0)
        y = y.astype(int)
        src = 'openml'
    except Exception:
        df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data', header=None)
        X = df.iloc[:, :-1].to_numpy(dtype=np.float32)
        y = df.iloc[:, -1].to_numpy(dtype=int)
        src = 'uci_csv'
    if np.isnan(X).any():
        med = np.nanmedian(X, axis=0)
        inds = np.where(np.isnan(X))
        X[inds] = med[inds[1]]
    return X, y, src

X, y, data_source = load_spambase_data()
idx_train_full, idx_test = train_test_split(np.arange(len(y)), test_size=TEST_SIZE, random_state=RNG, stratify=y)
idx_train, idx_valid = train_test_split(idx_train_full, test_size=VALID_SIZE_FROM_TRAIN, random_state=RNG, stratify=y[idx_train_full])
X_train, y_train = X[idx_train], y[idx_train]
X_valid, y_valid = X[idx_valid], y[idx_valid]
X_test, y_test = X[idx_test], y[idx_test]
X_train_valid, y_train_valid = X[idx_train_full], y[idx_train_full]
print(data_source, X_train.shape, X_valid.shape, X_test.shape)


In [ ]:
BASELINE_CANDIDATES = [
    dict(max_depth=5, eta=0.08, n_estimators=1600, min_child_weight=3, subsample=0.85, colsample_bytree=0.85, gamma=0.0, reg_alpha=0.0, reg_lambda=1.0),
    dict(max_depth=6, eta=0.06, n_estimators=2200, min_child_weight=3, subsample=0.90, colsample_bytree=0.90, gamma=0.0, reg_alpha=0.0, reg_lambda=1.0),
    dict(max_depth=4, eta=0.10, n_estimators=1200, min_child_weight=5, subsample=0.80, colsample_bytree=0.85, gamma=0.0, reg_alpha=0.0, reg_lambda=2.0),
]

def train_single_config(params, seed):
    xgb_params = {**BASE_OBJECTIVE_PARAMS, 'seed': int(seed), 'max_depth': int(params['max_depth']), 'eta': float(params['eta']), 'subsample': float(params['subsample']), 'colsample_bytree': float(params['colsample_bytree']), 'min_child_weight': float(params['min_child_weight']), 'gamma': float(params['gamma']), 'reg_alpha': float(params.get('reg_alpha',0.0)), 'reg_lambda': float(params.get('reg_lambda',1.0))}
    if 'colsample_bynode' in params:
        xgb_params['colsample_bynode'] = float(params['colsample_bynode'])
    dtrain = xgb.DMatrix(X_train, label=y_train); dvalid = xgb.DMatrix(X_valid, label=y_valid); dtest = xgb.DMatrix(X_test, label=y_test)
    model = xgb.train(params=xgb_params, dtrain=dtrain, num_boost_round=int(params['n_estimators']), evals=[(dtrain,'train'),(dvalid,'valid')], early_stopping_rounds=100, verbose_eval=False)
    best_round = int(getattr(model, 'best_iteration', int(params['n_estimators']) - 1) + 1)
    p_train = model.predict(dtrain, iteration_range=(0,best_round)); p_valid = model.predict(dvalid, iteration_range=(0,best_round)); p_test = model.predict(dtest, iteration_range=(0,best_round))
    metrics = dict(train_accuracy=accuracy_score(y_train, (p_train>=0.5).astype(int)), val_accuracy=accuracy_score(y_valid,(p_valid>=0.5).astype(int)), test_accuracy=accuracy_score(y_test,(p_test>=0.5).astype(int),), train_auc=roc_auc_score(y_train,p_train), val_auc=roc_auc_score(y_valid,p_valid), test_auc=roc_auc_score(y_test,p_test), train_logloss=log_loss(y_train,p_train), val_logloss=log_loss(y_valid,p_valid), test_logloss=log_loss(y_test,p_test), best_round=best_round)
    return model, metrics, xgb_params

def select_baseline():
    rows=[]
    for i,p in enumerate(BASELINE_CANDIDATES):
        vals=[]
        for s in BASELINE_SEEDS:
            _,m,_=train_single_config(p,s); vals.append(m['val_accuracy'])
        rows.append({'candidate_id':i,'params':p,'val_mean':float(np.mean(vals)),'val_std':float(np.std(vals))})
    bdf=pd.DataFrame(rows).sort_values(['val_mean','val_std'], ascending=[False,True]).reset_index(drop=True)
    return bdf.iloc[0]['params'], bdf

BASELINE_PARAMS, baseline_selection_df = select_baseline()
display(baseline_selection_df)


In [ ]:
def _unique_sorted(values, round_to=6):
    return sorted(set([round(float(v), round_to) for v in values]))

def extract_weightwatcher_metrics(details_df, matrix_kind):
    out = dict(matrix_kind=matrix_kind, alpha_primary=np.nan, alpha_mean=np.nan, alpha_min=np.nan, n_valid_layers=0, effective_rank=np.nan, rank=np.nan, n_rows=np.nan, n_cols=np.nan)
    if not isinstance(details_df, pd.DataFrame) or details_df.empty:
        return out
    alpha_vals = pd.to_numeric(details_df.get('alpha', np.nan), errors='coerce')
    tail_col = next((c for c in ['num_evals','num_pl_spikes','M'] if c in details_df.columns), None)
    valid = alpha_vals.notna() & np.isfinite(alpha_vals)
    if tail_col is not None:
        tail = pd.to_numeric(details_df[tail_col], errors='coerce')
        valid &= (tail.fillna(WW_MIN_TAIL) >= WW_MIN_TAIL)
    a = alpha_vals[valid]
    if len(a):
        out['alpha_primary'] = float(np.median(a)); out['alpha_mean'] = float(np.mean(a)); out['alpha_min'] = float(np.min(a)); out['n_valid_layers']=int(len(a))
    for c,n in [('effective_rank','effective_rank'),('rank','rank'),('N','n_rows'),('M','n_cols')]:
        if c in details_df.columns:
            out[n]=float(pd.to_numeric(details_df[c], errors='coerce').dropna().mean())
    return out

def run_config_across_seeds(sweep_name, sweep_value, params, seeds, search_stage, anchor_name):
    run_rows=[]
    for seed in seeds:
        model, metrics, train_params = train_single_config(params, seed)
        converted = convert(model=model, data=X_train_valid, labels=y_train_valid, W=W_MATRIX_PRIMARY, nfolds=5, t_points=40, random_state=RNG, train_params=train_params, num_boost_round=metrics['best_round'], multiclass='error', return_type='torch', verbose=False)
        details = ww.WeightWatcher(model=converted).analyze(**WW_ANALYZE_OPTIONS)
        ww_metrics = extract_weightwatcher_metrics(details, W_MATRIX_PRIMARY)
        row = {'configuration_id': f"{search_stage}::{sweep_name}::{sweep_value}", 'search_stage': search_stage, 'anchor_name': anchor_name, 'sweep_name': sweep_name, 'sweep_value': str(sweep_value), 'seed': int(seed), **params, **metrics, **ww_metrics}
        row['alpha_distance'] = abs(row['alpha_primary'] - TARGET_ALPHA) if pd.notna(row['alpha_primary']) else np.nan
        row['train_val_gap'] = row['train_accuracy'] - row['val_accuracy']
        row['train_test_gap'] = row['train_accuracy'] - row['test_accuracy']
        run_rows.append(row)
    return run_rows

def aggregate_results(per_seed_df):
    metric_cols=['train_accuracy','val_accuracy','test_accuracy','alpha_primary','alpha_distance','alpha_mean','alpha_min','effective_rank','train_val_gap','train_test_gap']
    gcols=['configuration_id','search_stage','anchor_name','sweep_name','sweep_value','max_depth','eta','n_estimators','min_child_weight','subsample','colsample_bytree','gamma','reg_alpha','reg_lambda']
    agg = per_seed_df.groupby(gcols, as_index=False).agg({c:['mean','std'] for c in metric_cols})
    agg.columns=['_'.join(c).strip('_') for c in agg.columns]
    return agg

def with_target_score(df, gap_reference=0.035):
    out=df.copy(); over=np.maximum(0.0, out['train_val_gap_mean']-gap_reference)
    out['target_score_mean']=out['val_accuracy_mean'] - 0.003*out['alpha_distance_mean'] - 0.25*over
    return out


## Stage 0: reproduce current notebook state and identify anchors


In [ ]:
def stage0_plan(base):
    n_base=int(base['n_estimators']); d_base=int(base['max_depth']); mcw=float(base['min_child_weight']); s_base=float(base['subsample']); eta=float(base['eta'])
    plan=[dict(sweep_name='baseline', sweep_value='baseline', params=dict(base))]
    for v in sorted(set(max(50, int(round(n_base*s))) for s in [0.25,0.5,0.75,1.0,1.25,1.5,2.0])):
        p=dict(base); p['n_estimators']=int(v); plan.append(dict(sweep_name='n_estimators', sweep_value=v, params=p))
    for v in sorted(set(int(np.clip(d_base+d,2,12)) for d in [-2,-1,0,1,2])):
        p=dict(base); p['max_depth']=int(v); plan.append(dict(sweep_name='max_depth', sweep_value=v, params=p))
    for v in sorted(set(max(1.0, round(mcw*s,3)) for s in [0.25,0.5,1.0,2.0,4.0])):
        p=dict(base); p['min_child_weight']=float(v); plan.append(dict(sweep_name='min_child_weight', sweep_value=v, params=p))
    for v in _unique_sorted([max(0.4,s_base-0.30), max(0.5,s_base-0.15), s_base, min(1.0,s_base+0.10), 1.0],3):
        p=dict(base); p['subsample']=float(v); plan.append(dict(sweep_name='subsample', sweep_value=v, params=p))
    budget=eta*n_base
    for e in sorted(set(float(np.clip(eta*s,0.01,0.3)) for s in [0.5,1.0,2.0])):
        n=max(100,int(round(budget/e))); p=dict(base); p['eta']=e; p['n_estimators']=n
        plan.append(dict(sweep_name='eta_x_n_estimators', sweep_value=f'eta={e:.4f}|n={n}', params=p))
    return plan

all_rows=[]
for item in stage0_plan(BASELINE_PARAMS):
    all_rows.extend(run_config_across_seeds(item['sweep_name'], item['sweep_value'], item['params'], BASELINE_SEEDS, 'stage0', 'ACCURACY_BASELINE'))

per_seed_results_df = pd.DataFrame(all_rows)
aggregated_results_df = aggregate_results(per_seed_results_df)
ACCURACY_BASELINE = aggregated_results_df.sort_values('val_accuracy_mean', ascending=False).iloc[0]
within_tol = aggregated_results_df[aggregated_results_df['val_accuracy_mean'] >= ACCURACY_BASELINE['val_accuracy_mean'] - VAL_ACC_TOL].copy()
if within_tol.empty:
    within_tol = aggregated_results_df[aggregated_results_df['val_accuracy_mean'] >= aggregated_results_df['val_accuracy_mean'].quantile(0.90)]
ALPHA_FRIENDLY_BASELINE = within_tol.sort_values(['alpha_distance_mean','val_accuracy_mean'], ascending=[True,False]).iloc[0]
gap_reference = float(ALPHA_FRIENDLY_BASELINE['train_val_gap_mean']) if pd.notna(ALPHA_FRIENDLY_BASELINE['train_val_gap_mean']) else 0.035
aggregated_results_df = with_target_score(aggregated_results_df, gap_reference)
print('ACCURACY_BASELINE:', ACCURACY_BASELINE['configuration_id'])
print('ALPHA_FRIENDLY_BASELINE:', ALPHA_FRIENDLY_BASELINE['configuration_id'])


## Stage 1 + Stage 2 + Stage 3


In [ ]:
def clip(v, lo, hi): return max(lo, min(hi, v))

def build_stage1(anchor):
    a = {k: anchor[k] for k in ['max_depth','eta','n_estimators','min_child_weight','subsample','colsample_bytree','gamma','reg_alpha','reg_lambda']}
    d=int(a['max_depth']); mcw=float(a['min_child_weight']); eta=float(a['eta']); n0=int(a['n_estimators'])
    plan=[]
    for v in _unique_sorted([0.90,0.95,0.975,1.00,a['subsample']],3): p=dict(a); p['subsample']=v; plan.append(dict(sweep_name='stage1_subsample', sweep_value=v, params=p))
    for v in _unique_sorted([0.90,0.95,1.00,a['colsample_bytree']],3): p=dict(a); p['colsample_bytree']=v; plan.append(dict(sweep_name='stage1_colsample_bytree', sweep_value=v, params=p))
    for v in sorted(set([max(3,d-2),max(3,d-1),d,min(6,d+1)])): p=dict(a); p['max_depth']=int(v); plan.append(dict(sweep_name='stage1_max_depth', sweep_value=v, params=p))
    for v in sorted(set([clip(max(4,mcw),4,64),clip(max(8,2*mcw),4,64),clip(max(16,4*mcw),4,64)])): p=dict(a); p['min_child_weight']=float(v); plan.append(dict(sweep_name='stage1_min_child_weight', sweep_value=v, params=p))
    for v in [0.0,0.5,1.0,2.0]: p=dict(a); p['gamma']=v; plan.append(dict(sweep_name='stage1_gamma', sweep_value=v, params=p))
    for v in [1.0,3.0,10.0]: p=dict(a); p['reg_lambda']=v; plan.append(dict(sweep_name='stage1_reg_lambda', sweep_value=v, params=p))
    budget=eta*n0
    for e in sorted(set([clip(v,0.015,0.12) for v in [0.02,0.03,0.05,0.08,eta]])):
        nominal=max(100,int(round(budget/e)))
        for m in [0.8,1.0,1.2]:
            n=max(100,int(round(nominal*m))); p=dict(a); p['eta']=e; p['n_estimators']=n; plan.append(dict(sweep_name='stage1_eta_x_n_estimators', sweep_value=f'eta={e:.4f}|n={n}', params=p))
    return list({tuple(sorted(x['params'].items())):x for x in plan}.values())

for item in build_stage1(ALPHA_FRIENDLY_BASELINE):
    all_rows.extend(run_config_across_seeds(item['sweep_name'], item['sweep_value'], item['params'], BASELINE_SEEDS[:N_COARSE_SEEDS], 'stage1', 'ALPHA_FRIENDLY_BASELINE'))

per_seed_results_df = pd.DataFrame(all_rows)
aggregated_results_df = with_target_score(aggregate_results(per_seed_results_df), gap_reference)
best_val = aggregated_results_df['val_accuracy_mean'].max()
pareto = aggregated_results_df[(aggregated_results_df['val_accuracy_mean'] >= best_val - 0.003) & (aggregated_results_df['alpha_distance_mean'] <= 0.50)].sort_values('target_score_mean', ascending=False).head(8)

refine=[]
for _,r in pareto.iterrows():
    b=r[['max_depth','eta','n_estimators','min_child_weight','subsample','colsample_bytree','gamma','reg_alpha','reg_lambda']].to_dict(); budget=b['eta']*b['n_estimators']
    for d in sorted(set([int(clip(b['max_depth']+x,3,6)) for x in [-1,0,1]])): p=dict(b); p['max_depth']=d; refine.append(dict(sweep_name='refine_max_depth', sweep_value=f'd={d}', params=p))
    for m in [0.5,1.0,2.0]: p=dict(b); p['min_child_weight']=float(clip(b['min_child_weight']*m,4,64)); refine.append(dict(sweep_name='refine_mcw', sweep_value=f'm={m}', params=p))
    for s in [0.0,0.025,0.05]: p=dict(b); p['subsample']=float(clip(b['subsample']+s,0.5,1.0)); refine.append(dict(sweep_name='refine_subsample', sweep_value=f's={s}', params=p))
    for s in [0.0,0.025,0.05]: p=dict(b); p['colsample_bytree']=float(clip(b['colsample_bytree']+s,0.5,1.0)); refine.append(dict(sweep_name='refine_colsample', sweep_value=f'c={s}', params=p))
    for m in [0.8,1.0,1.2]:
        e=float(clip(b['eta']*m,0.01,0.2)); nominal=max(100,int(round(budget/e)))
        for adj in [0.9,1.0,1.1]:
            n=max(100,int(round(nominal*adj))); p=dict(b); p['eta']=e; p['n_estimators']=n; refine.append(dict(sweep_name='refine_eta_n', sweep_value=f'e={e}|n={n}', params=p))

for item in list({tuple(sorted(x['params'].items())):x for x in refine}.values()):
    all_rows.extend(run_config_across_seeds(item['sweep_name'], item['sweep_value'], item['params'], list(range(N_REFINE_SEEDS)), 'stage2', 'PARETO_REFINEMENT'))

per_seed_results_df = pd.DataFrame(all_rows)
aggregated_results_df = with_target_score(aggregate_results(per_seed_results_df), gap_reference)
agg=aggregated_results_df
best_validation_model = agg.sort_values('val_accuracy_mean', ascending=False).iloc[0]
hard = agg[(agg['alpha_primary_mean']>=HARD_ALPHA_BAND[0]) & (agg['alpha_primary_mean']<=HARD_ALPHA_BAND[1])]
soft = agg[(agg['alpha_primary_mean']>=SOFT_ALPHA_BAND[0]) & (agg['alpha_primary_mean']<=SOFT_ALPHA_BAND[1])]
if len(hard): best_alpha2_model = hard.sort_values('val_accuracy_mean', ascending=False).iloc[0]
elif len(soft): best_alpha2_model = soft.sort_values('val_accuracy_mean', ascending=False).iloc[0]
else:
    near = agg[agg['val_accuracy_mean'] >= best_validation_model['val_accuracy_mean'] - VAL_ACC_TOL]
    best_alpha2_model = near.sort_values('alpha_distance_mean').iloc[0] if len(near) else agg.sort_values('alpha_distance_mean').iloc[0]
best_pareto_model = agg.sort_values('target_score_mean', ascending=False).iloc[0]

ids=set([ACCURACY_BASELINE['configuration_id'], ALPHA_FRIENDLY_BASELINE['configuration_id'], best_validation_model['configuration_id'], best_alpha2_model['configuration_id'], best_pareto_model['configuration_id']])
final_shortlist = agg[agg['configuration_id'].isin(ids)].copy()
final_shortlist['is_accuracy_baseline'] = final_shortlist['configuration_id'].eq(ACCURACY_BASELINE['configuration_id'])
final_shortlist['is_alpha_friendly_baseline'] = final_shortlist['configuration_id'].eq(ALPHA_FRIENDLY_BASELINE['configuration_id'])
final_shortlist['is_best_validation_model'] = final_shortlist['configuration_id'].eq(best_validation_model['configuration_id'])
final_shortlist['is_best_alpha2_model'] = final_shortlist['configuration_id'].eq(best_alpha2_model['configuration_id'])
final_shortlist['is_best_pareto_model'] = final_shortlist['configuration_id'].eq(best_pareto_model['configuration_id'])
display(final_shortlist)



In [ ]:
def make_plots(df):
    fig,ax=plt.subplots(2,4,figsize=(20,9))
    ax[0,0].scatter(df['alpha_primary_mean'], df['val_accuracy_mean']); ax[0,0].axvline(TARGET_ALPHA,color='r',ls='--'); ax[0,0].set_title('val acc vs alpha')
    ax[0,1].scatter(df['alpha_primary_mean'], df['test_accuracy_mean']); ax[0,1].axvline(TARGET_ALPHA,color='r',ls='--'); ax[0,1].set_title('test acc vs alpha')
    ax[0,2].scatter(df['alpha_distance_mean'], df['val_accuracy_mean']); ax[0,2].axvline(0.0,color='r',ls='--'); ax[0,2].set_title('val acc vs alpha_distance')
    ax[0,3].scatter(df['alpha_distance_mean'], df['test_accuracy_mean']); ax[0,3].axvline(0.0,color='r',ls='--'); ax[0,3].set_title('test acc vs alpha_distance')
    ax[1,0].scatter(df['alpha_distance_mean'], df['train_val_gap_mean']); ax[1,0].axvline(0.0,color='r',ls='--'); ax[1,0].set_title('train-val gap vs alpha_distance')
    ax[1,1].scatter(df['alpha_distance_mean'], df['val_accuracy_mean']); ax[1,1].set_title('Pareto')
    ax[1,2].scatter(df['alpha_distance_mean'], df['val_accuracy_mean']); ax[1,2].axvspan(0,HARD_ALPHA_BAND[1]-TARGET_ALPHA,color='g',alpha=0.2); ax[1,2].set_title('Pareto + hard band')
    ax[1,3].scatter(df['alpha_primary_mean'], df['val_accuracy_mean'], alpha=0.35)
    for name,row in [('ACCURACY_BASELINE',ACCURACY_BASELINE),('ALPHA_FRIENDLY_BASELINE',ALPHA_FRIENDLY_BASELINE),('best_validation_model',best_validation_model),('best_alpha2_model',best_alpha2_model),('best_pareto_model',best_pareto_model)]:
        ax[1,3].scatter([row['alpha_primary_mean']],[row['val_accuracy_mean']],marker='*',s=180,label=name)
    ax[1,3].axvline(TARGET_ALPHA,color='r',ls='--'); ax[1,3].legend(fontsize=7); ax[1,3].set_title('final highlight')
    plt.tight_layout(); plt.show()

make_plots(agg)



## Why alpha_distance matters more than raw alpha here

Raw alpha is still reported for transparency, but distance to the target regime drives targeted ranking.


## What counts as success

1. A model inside HARD_ALPHA_BAND with competitive validation/test accuracy.
2. A clear Pareto frontier with negligible validation loss as alpha moves toward 2.0.
3. If raw W7 misses the regime, secondary matrix diagnostics on final shortlist can show cleaner alpha behavior.


In [ ]:
need_secondary = not ((agg['alpha_primary_mean'] <= 2.3) & (agg['val_accuracy_mean'] >= best_validation_model['val_accuracy_mean'] - 0.002)).any()
if need_secondary:
    print('Secondary diagnostics trigger condition met.')
    if RUN_SECONDARY_W8:
        print('W8 diagnostics can be run on shortlist configs in this block.')
    else:
        print('RUN_SECONDARY_W8=False; secondary diagnostics skipped by default.')



In [ ]:
per_seed_results_df.to_csv(RESULTS_DIR / 'per_seed_results.csv', index=False)
aggregated_results_df.to_csv(RESULTS_DIR / 'aggregated_results.csv', index=False)
final_shortlist.to_csv(RESULTS_DIR / 'final_shortlist.csv', index=False)
print('Saved:', RESULTS_DIR)
